In [1]:
import argparse,sys, os
import copy
import time
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.optim as optim
from sklearn.model_selection import LeavePGroupsOut
import random
import pickle
import torch.optim.lr_scheduler as lr_Scheduler
from torchsummary import summary
from models import getModels
import mne

In [2]:
class Concatdataset(torch.utils.data.Dataset):
    def __init__(self, *datasets):
        self.datasets = datasets

    def __getitem__(self, i):
        return tuple(d[i] for d in self.datasets)

    def __len__(self):
        return min(len(d) for d in self.datasets)


def modify_dataset(filename):
    #need to modify the dataset as data from subj 21,22 missing

    f = h5py.File(filename,'r+')
    group = f["home"]
    data = group["data"]
    subj_labels = group["subj_labels"]

    for i in range(data.shape[0]):
        if subj_labels[i]>20:
            subj_labels[i]=subj_labels[i]-2
                
    f.close()
    
def modifyfft(data):
    tdata = torch.zeros(data.shape)
    for i in range(data.shape[0]):
        t = data[i].T
        ndata=mne.filter.filter_data(t,200, l_freq=None, h_freq=40, picks=None, filter_length='auto',h_trans_bandwidth=2,verbose=False)
        ndata = torch.tensor(ndata)
        ndata = ndata.permute(1,0)
        tdata[i]=ndata
 
    print("data transform complete",flush=True)
    return tdata

def load_split_dataset(filename,n_groups_testing,validation_fold,random_seed):

    f = h5py.File(filename,'r')
    group = f["home"]
    data = group["data"]
    subj_labels = group["subj_labels"]
    labels = torch.zeros(data.shape[0])
    group_num = torch.zeros(data.shape[0])
    subj_labels_mod = torch.zeros(data.shape[0])

    print("data loading done")
    total_subj = 36
    n_dreamers = 18
    n_nondreamers = 18
    for i in range(data.shape[0]):
        t = subj_labels[i]
        if t<=n_dreamers:
            labels[i]=0
            group_num[i]=t
        else:
            labels[i]=1
            group_num[i]=t-18 
        subj_labels_mod[i] = subj_labels[i]-1


    data = modifyfft(data)
    main_dataset = Concatdataset(data,subj_labels_mod,labels)

    lpgo = LeavePGroupsOut(n_groups=n_groups_testing)
    all_splits=lpgo.split(data,groups=group_num)
    selected_splits=[]
    random.seed(random_seed)
    selection_array= random.sample(range(0, lpgo.get_n_splits(data,groups=group_num)),validation_fold)
    j=0
    for t in all_splits:
        if j in selection_array:
            selected_splits.append(t)
        j=j+1
    
    print("data split done")

    return main_dataset,selected_splits



In [3]:
filename= '../data/all_AWA.h5'
n_groups_testing= 2
validation_fold= 5
save_model= False
batch_size= 64
num_epochs= 100
learning_rate= 0.00005
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random_seed = 42
validation_set=0
model_num = 1
print_every = 1

In [4]:
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

In [5]:
main_dataset,selected_splits = load_split_dataset(filename,n_groups_testing,validation_fold,random_seed)
selected_splits=[selected_splits[validation_set]]
train_indices = selected_splits[0][0]
val_indices = selected_splits[0][1]

data loading done
data transform complete
data split done


In [6]:
train_sampler = Data.SubsetRandomSampler(train_indices)
valid_sampler = Data.SubsetRandomSampler(val_indices)

train_loader = torch.utils.data.DataLoader(main_dataset, batch_size=batch_size, 
                                                   sampler=train_sampler)
validation_loader = torch.utils.data.DataLoader(main_dataset, batch_size=batch_size,
                                                        sampler=valid_sampler)    

In [7]:
extractor,classifier = getModels(model_num)

In [8]:
extractor.to(device)
classifier.to(device)
criterion = nn.NLLLoss()
params = list(extractor.parameters()) + list(classifier.parameters())
optimizer = optim.Adam(params, lr=learning_rate)

In [9]:
best_accuracy = 0.0
best_accuracy_epoch=0
best_model_ex_wts = copy.deepcopy(extractor.state_dict())
best_model_cl_wts = copy.deepcopy(classifier.state_dict())

In [10]:
for epoch in range(num_epochs):
    
    # train model
    for batchid,traindata in enumerate(train_loader):
        source_eeg,source_subj,source_labels = traindata
        
        extractor.train()
        classifier.train()
        optimizer.zero_grad()
        
        #data_shape = (batch_size,1000,19) : reshape to fit cnn input
        source_eeg = source_eeg.float().to(device=device)
        source_eeg = source_eeg.reshape(source_eeg.shape[0],1,source_eeg.shape[1],source_eeg.shape[2])
        source_labels=source_labels.long().to(device=device)
        
        
        extractor_out = extractor(source_eeg)
        classifier_out = classifier(extractor_out)
        loss = criterion(classifier_out, source_labels)
        loss.backward()
        optimizer.step()
        
    # test and print 
    if (epoch+1)%print_every==0:
        extractor.eval()
        classifier.eval()
        
        # compute loss, accuracy on test set
        test_accuracy=0.0
        test_loss=0.0
        
        with torch.no_grad():
            for batchidx,[eeg_data,subj_labels,dreamer_labels] in enumerate(validation_loader):

                            
                eeg_data = eeg_data.float().to(device=device)
                eeg_data = eeg_data.reshape(eeg_data.shape[0],1,eeg_data.shape[1],eeg_data.shape[2])
                dreamer_labels=dreamer_labels.long().to(device=device)
                extractor_out = extractor(eeg_data)
                classifier_out = classifier(extractor_out)
                            
                probs = torch.exp(classifier_out)
                top_p, top_class = probs.topk(1, dim=1)
                loss = criterion(classifier_out, dreamer_labels)
                test_loss += loss.item()
                            
                            
                equals = top_class == dreamer_labels.view(*top_class.shape)
                test_accuracy += torch.mean(equals.type(torch.FloatTensor)).item()


        test_accuracy=test_accuracy/len(validation_loader)
        test_loss=test_loss/len(validation_loader)
        
        # store best model weights
        
        if(epoch>5):
            if(test_accuracy>best_accuracy):
                best_accuracy=test_accuracy
                best_accuracy_epoch=epoch
                best_model_ex_wts = copy.deepcopy(extractor.state_dict())
                best_model_cl_wts = copy.deepcopy(classifier.state_dict())
        
        # compute loss, accuracy on train set
        
        train_accuracy=0.0
        train_loss=0.0
        with torch.no_grad():
            for batchidx,[eeg_data,subj_labels,dreamer_labels] in enumerate(train_loader):
  
                eeg_data = eeg_data.float().to(device=device)
                eeg_data = eeg_data.reshape(eeg_data.shape[0],1,eeg_data.shape[1],eeg_data.shape[2])
                        
                dreamer_labels=dreamer_labels.long().to(device=device)
                            
                extractor_out = extractor(eeg_data)
                classifier_out = classifier(extractor_out)
                            
                probs = torch.exp(classifier_out)
                top_p, top_class = probs.topk(1, dim=1)
                loss = criterion(classifier_out, dreamer_labels)
                train_loss += loss.item()
                                    
                equals = top_class == dreamer_labels.view(*top_class.shape)
                train_accuracy += torch.mean(equals.type(torch.FloatTensor)).item()
                            

        train_accuracy=train_accuracy/len(train_loader)
        train_loss=train_loss/len(train_loader)
            
        print('epoch: %d/%d train loss: %.3f  test loss: %.3f test_accuracy: %.3f best_accuracy: %.3f'%(epoch+1,num_epochs,train_loss,test_loss,test_accuracy,best_accuracy),flush=True)
                

epoch: 1/100 train loss: 0.700  test loss: 0.703 test_accuracy: 0.822 best_accuracy: 0.000
epoch: 2/100 train loss: 0.695  test loss: 0.697 test_accuracy: 0.818 best_accuracy: 0.000
epoch: 3/100 train loss: 0.694  test loss: 0.706 test_accuracy: 0.814 best_accuracy: 0.000
epoch: 4/100 train loss: 0.694  test loss: 0.714 test_accuracy: 0.809 best_accuracy: 0.000
epoch: 5/100 train loss: 0.695  test loss: 0.725 test_accuracy: 0.778 best_accuracy: 0.000
epoch: 6/100 train loss: 0.686  test loss: 0.761 test_accuracy: 0.632 best_accuracy: 0.000
epoch: 7/100 train loss: 0.653  test loss: 0.994 test_accuracy: 0.344 best_accuracy: 0.344
epoch: 8/100 train loss: 0.588  test loss: 1.407 test_accuracy: 0.257 best_accuracy: 0.344
epoch: 9/100 train loss: 0.566  test loss: 1.413 test_accuracy: 0.244 best_accuracy: 0.344
epoch: 10/100 train loss: 0.555  test loss: 1.421 test_accuracy: 0.241 best_accuracy: 0.344
epoch: 11/100 train loss: 0.500  test loss: 1.190 test_accuracy: 0.275 best_accuracy: 0.3

epoch: 91/100 train loss: 0.115  test loss: 0.757 test_accuracy: 0.644 best_accuracy: 0.802
epoch: 92/100 train loss: 0.117  test loss: 0.874 test_accuracy: 0.594 best_accuracy: 0.802
epoch: 93/100 train loss: 0.096  test loss: 0.450 test_accuracy: 0.804 best_accuracy: 0.804
epoch: 94/100 train loss: 0.166  test loss: 1.231 test_accuracy: 0.423 best_accuracy: 0.804
epoch: 95/100 train loss: 0.104  test loss: 0.734 test_accuracy: 0.650 best_accuracy: 0.804
epoch: 96/100 train loss: 0.094  test loss: 0.492 test_accuracy: 0.778 best_accuracy: 0.804
epoch: 97/100 train loss: 0.134  test loss: 1.184 test_accuracy: 0.478 best_accuracy: 0.804
epoch: 98/100 train loss: 0.107  test loss: 0.616 test_accuracy: 0.717 best_accuracy: 0.804
epoch: 99/100 train loss: 0.087  test loss: 0.521 test_accuracy: 0.765 best_accuracy: 0.804
epoch: 100/100 train loss: 0.086  test loss: 0.679 test_accuracy: 0.690 best_accuracy: 0.804


In [14]:
save_model = True

In [15]:
if save_model == True:
    store_dir_name = './models/'
    tname=store_dir_name+'validation_set_{}_model.pth'.format(validation_set)
    torch.save({
                'ex_state_dict': best_model_ex_wts,
                'cl_state_dict': best_model_cl_wts,
                'optimizer_state_dict': optimizer.state_dict(),
            }, tname) 